In [0]:
%run ../utils/utils_config

In [0]:
# mirror your S3 partition structure in Volumes
# your S3 path was: plant_id=Plant_1/year=2026/month=06/day=04/hour=02/

landing_partition = (
    f'{LANDING_PATH}'
    f'plant_id=Plant_1/'
    f'year=2026/'
    f'month=06/'
    f'day=04/'
    f'hour=02/'
)

dbutils.fs.mkdirs(landing_partition)
print(f'Landing zone ready:\n{landing_partition}')

In [0]:
# verify file landed correctly
files = dbutils.fs.ls(landing_partition)
for f in files:
    print(f'File : {f.name}')
    print(f'Size : {round(f.size/1024, 1)} KB')

In [0]:
# read parquet and inspect
df_raw = spark.read.parquet(landing_partition)

print('=== Schema ===')
df_raw.printSchema()

print(f'\n=== Row count : {df_raw.count()} ===')
print(f'=== Turbines  : {df_raw.select("asset_id").distinct().count()} ===')

display(df_raw)

In [0]:
# confirm all 10 turbines present
print('Turbines in file:')
df_raw.select('asset_id').distinct().orderBy('asset_id').show()

# check timestamp range
from pyspark.sql.functions import min, max
df_raw.select(
    min('timestamp_utc').alias('earliest'),
    max('timestamp_utc').alias('latest')
).show(truncate=False)

In [0]:
# read from landing
df_raw = spark.read.parquet(
    '/Volumes/workspace/wind_turbine/landing/plant_id=Plant_1/year=2026/month=06/day=04/hour=02/'
)

print('Schema:')
df_raw.printSchema()
print(f'Row count: {df_raw.count()}')

In [0]:
# write as Unity Catalog managed Delta table
# no LOCATION needed — Unity Catalog manages storage automatically
df_raw.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.wind_turbine.bronze')

print('Bronze table created ✓')

In [0]:
# verify it worked
spark.sql("SELECT COUNT(*) as row_count FROM workspace.wind_turbine.bronze").show()
spark.sql("DESCRIBE DETAIL workspace.wind_turbine.bronze").show(truncate=False)

In [0]:
# check all turbines present
spark.sql("""
    SELECT 
        asset_id,
        COUNT(*) as row_count,
        MIN(timestamp_utc) as earliest,
        MAX(timestamp_utc) as latest
    FROM workspace.wind_turbine.bronze
    GROUP BY asset_id
    ORDER BY asset_id
""").show(truncate=False)